In [7]:
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

train_df = pd.read_csv("preprocessed_train_final.csv")
test_df  = pd.read_csv("preprocessed_test_final.csv")


C:\Users\ilker\AppData\Local\Temp\ipykernel_28268\622950258.py:6: DtypeWarning: Columns (0: product_id) have mixed types. Specify dtype option on import or set low_memory=False.
  test_df  = pd.read_csv("preprocessed_test_final.csv")


In [8]:
train_df = train_df.groupby('category_fixed').sample(n=1).reset_index(drop=True)
test_df = test_df.groupby('category').sample(n=1).reset_index(drop=True)

In [9]:
import json
category_mapping = json.load(open('category_map.json', 'r', encoding='utf-8'))


In [10]:
import torch
from datasets import Dataset
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer
)

In [12]:

import numpy as np
import pandas as pd
import torch

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, f1_score

In [13]:
TEXT_COL = "text_for_model"  # <-- change if needed
LEAF_TRAIN_COL = "category_fixed" if "category_fixed" in train_df.columns else "category"
LEAF_TEST_COL  = "category_fixed" if "category_fixed" in test_df.columns else "category"

# sample_weight optional
if "sample_weight" not in train_df.columns:
    train_df["sample_weight"] = 1.0
if "sample_weight" not in test_df.columns:
    test_df["sample_weight"] = 1.0

# main labels from leaf
train_df["main_true"] = train_df[LEAF_TRAIN_COL].astype(str).map(category_mapping).fillna("Diğer")
test_df["main_true"]  = test_df[LEAF_TEST_COL].astype(str).map(category_mapping).fillna("Diğer")

print("Train:", train_df.shape, "| Test:", test_df.shape)
print("Leaf:", LEAF_TRAIN_COL, LEAF_TEST_COL)
train_df[[TEXT_COL, LEAF_TRAIN_COL, "main_true", "sample_weight"]].head()


Train: (1140, 37) | Test: (1140, 14)
Leaf: category_fixed category


,text_for_model,category_fixed,main_true,sample_weight
0,300mbps 3x5dbi anten n access point,ADSL Modemler,Elektronik,1.0
1,modern dizayn büyük aslan abajur bakır bej,Abajur,Ev & Yaşam,1.0
2,zümrüt midi bacak dekolteli uzun kol davet elb...,Abiye & Mezuniyet Elbisesi,Giyim,1.0
3,12cm sedef deri kırık beyaz gelin ayakkabısı g...,Abiye Ayakkabı,Ayakkabı,1.0
4,kadın fuşya taş detaylı abiye el çantası zinci...,Abiye Çanta,Aksesuar,1.0


In [14]:
def topk_idx_from_logits(logits: np.ndarray, k: int) -> np.ndarray:
    # logits: (N, C)
    idx = np.argpartition(-logits, kth=k-1, axis=1)[:, :k]
    row = np.arange(logits.shape[0])[:, None]
    idx_sorted = idx[row, np.argsort(-logits[row, idx], axis=1)]
    return idx_sorted

def topk_accuracy(y_true: np.ndarray, topk_idx: np.ndarray) -> float:
    # topk_idx: (N, k)
    return float(np.mean([y_true[i] in topk_idx[i] for i in range(len(y_true))]))

def compute_leaf_metrics_from_logits(logits: np.ndarray, y_true: np.ndarray) -> dict:
    pred = np.argmax(logits, axis=1)
    top3 = topk_idx_from_logits(logits, 3)
    top5 = topk_idx_from_logits(logits, 5)
    return {
        "acc": float(accuracy_score(y_true, pred)),
        "macro_f1": float(f1_score(y_true, pred, average="macro")),
        "weighted_f1": float(f1_score(y_true, pred, average="weighted")),
        "top3_acc": float(topk_accuracy(y_true, top3)),
        "top5_acc": float(topk_accuracy(y_true, top5)),
    }


In [15]:
class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.get("labels")
        weights = inputs.get("sample_weight", None)

        outputs = model(
            input_ids=inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
            token_type_ids=inputs.get("token_type_ids", None),
            labels=None
        )
        logits = outputs.logits
        loss_vec = torch.nn.functional.cross_entropy(logits, labels, reduction="none")

        if weights is not None:
            weights = weights.to(loss_vec.dtype)
            loss = (loss_vec * weights).mean()
        else:
            loss = loss_vec.mean()

        return (loss, outputs) if return_outputs else loss


In [16]:
def make_hf_dataset(df: pd.DataFrame, text_col: str, label_col: str, label2id: dict, tokenizer, max_len: int):
    tmp = df[[text_col, label_col, "sample_weight"]].copy()
    tmp[text_col] = tmp[text_col].fillna("").astype(str)
    tmp["label"] = tmp[label_col].astype(str).map(label2id).astype(int)

    ds = Dataset.from_pandas(tmp[[text_col, "label", "sample_weight"]], preserve_index=False)

    def tok(batch):
        return tokenizer(batch[text_col], truncation=True, padding="max_length", max_length=max_len)

    ds = ds.map(tok, batched=True, remove_columns=[text_col])
    ds.set_format(type="torch")
    return ds


In [17]:
torch.backends.cuda.matmul.allow_tf32 = True

BASE_MODEL = "Trendyol/tyroberta"  # good for TR+EN mix
tokenizer_main = AutoTokenizer.from_pretrained(BASE_MODEL)

# label encoders for main
main_labels = sorted(train_df["main_true"].unique().tolist())
main_le = LabelEncoder().fit(main_labels)

main_id2label = {i: lab for i, lab in enumerate(main_le.classes_)}
main_label2id = {lab: i for i, lab in enumerate(main_le.classes_)}

# build datasets
MAX_LEN_MAIN = 96
ds_main_train = make_hf_dataset(
    train_df, TEXT_COL, "main_true", main_label2id, tokenizer_main, MAX_LEN_MAIN
)
ds_main_test = make_hf_dataset(
    test_df, TEXT_COL, "main_true", main_label2id, tokenizer_main, MAX_LEN_MAIN
)

model_main = AutoModelForSequenceClassification.from_pretrained(
    BASE_MODEL,
    num_labels=len(main_label2id),
    id2label=main_id2label,
    label2id=main_label2id
)

args_main = TrainingArguments(
    output_dir="bert_stage1_main",
    learning_rate=2e-5,
    per_device_train_batch_size=64,     # A100 friendly; if OOM -> 32
    per_device_eval_batch_size=128,
    num_train_epochs=1,
    fp16=True,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    logging_steps=200,
    report_to="none"
)

trainer_main = WeightedTrainer(
    model=model_main,
    args=args_main,
    train_dataset=ds_main_train,
    eval_dataset=ds_main_test,
    tokenizer=tokenizer_main
)

trainer_main.train()
pred_main = trainer_main.predict(ds_main_test)
main_logits_test = pred_main.predictions
y_main_test = pred_main.label_ids

# main metrics (Top-1 + macroF1)
main_pred = np.argmax(main_logits_test, axis=1)
print("MAIN acc:", accuracy_score(y_main_test, main_pred))
print("MAIN macroF1:", f1_score(y_main_test, main_pred, average="macro"))


c:\Users\ilker\anaconda3\envs\torchgpu\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\ilker\.cache\huggingface\hub\models--bert-base-multilingual-cased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Map: 100%|██████████| 1140/1140 [00:00<00:00, 4430.81 examples/s]
Some weights of BertForSequenceClass

Epoch,Training Loss,Validation Loss
1,No log,1.823901


MAIN acc: 0.44298245614035087
MAIN macroF1: 0.06139817629179332


In [18]:
def predict_top2_main(trainer, ds):
    out = trainer.predict(ds)
    logits = out.predictions
    top2 = topk_idx_from_logits(logits, 2)
    top1_ids = top2[:, 0]
    top2_ids = top2[:, 1]

    # convert ids -> names
    top1_lab = np.array([main_id2label[i] for i in top1_ids], dtype=object)
    top2_lab = np.array([main_id2label[i] for i in top2_ids], dtype=object)

    # probs (optional)
    probs = torch.softmax(torch.tensor(logits), dim=1).cpu().numpy()
    p1 = probs[np.arange(len(probs)), top1_ids].astype(np.float32)
    p2 = probs[np.arange(len(probs)), top2_ids].astype(np.float32)

    return top1_lab, top2_lab, p1, p2

# predict for train and test
train_main_top1, train_main_top2, train_p1, train_p2 = predict_top2_main(trainer_main, ds_main_train)
test_main_top1,  test_main_top2,  test_p1,  test_p2  = predict_top2_main(trainer_main, ds_main_test)

train_df["main_pred1"] = train_main_top1
train_df["main_pred2"] = train_main_top2
test_df["main_pred1"]  = test_main_top1
test_df["main_pred2"]  = test_main_top2

def add_main_tokens(df: pd.DataFrame, base_text_col: str) -> pd.Series:
    return (
        df[base_text_col].fillna("").astype(str)
        + " __MAIN1__ " + df["main_pred1"].astype(str)
        + " __MAIN2__ " + df["main_pred2"].astype(str)
    )

train_df["text_plus_main2"] = add_main_tokens(train_df, TEXT_COL)
test_df["text_plus_main2"]  = add_main_tokens(test_df, TEXT_COL)

train_df[["main_pred1","main_pred2","text_plus_main2"]].head(3)


,main_pred1,main_pred2,text_plus_main2
0,Ev & Yaşam,Elektronik,300mbps 3x5dbi anten n access point __MAIN1__ ...
1,Ev & Yaşam,Giyim,modern dizayn büyük aslan abajur bakır bej __M...
2,Ev & Yaşam,Giyim,zümrüt midi bacak dekolteli uzun kol davet elb...


In [19]:
# leaf labels
leaf_le = LabelEncoder()
leaf_le.fit(train_df[LEAF_TRAIN_COL].astype(str).values)

leaf_id2label = {i: lab for i, lab in enumerate(leaf_le.classes_)}
leaf_label2id = {lab: i for i, lab in enumerate(leaf_le.classes_)}

MAX_LEN_LEAF = 96  # usually enough for your title lengths + main tokens

tokenizer_leaf = AutoTokenizer.from_pretrained(BASE_MODEL)

# datasets for baseline (text only)
ds_leaf_train_base = make_hf_dataset(
    train_df, TEXT_COL, LEAF_TRAIN_COL, leaf_label2id, tokenizer_leaf, MAX_LEN_LEAF
)
ds_leaf_test_base = make_hf_dataset(
    test_df, TEXT_COL, LEAF_TEST_COL, leaf_label2id, tokenizer_leaf, MAX_LEN_LEAF
)

# datasets for main-augmented (text + __MAIN1__/__MAIN2__)
ds_leaf_train_main = make_hf_dataset(
    train_df, "text_plus_main2", LEAF_TRAIN_COL, leaf_label2id, tokenizer_leaf, MAX_LEN_LEAF
)
ds_leaf_test_main = make_hf_dataset(
    test_df, "text_plus_main2", LEAF_TEST_COL, leaf_label2id, tokenizer_leaf, MAX_LEN_LEAF
)


Map: 100%|██████████| 1140/1140 [00:00<00:00, 14376.75 examples/s]


In [20]:
model_leaf_base = AutoModelForSequenceClassification.from_pretrained(
    BASE_MODEL,
    num_labels=len(leaf_label2id),
    id2label=leaf_id2label,
    label2id=leaf_label2id
)

args_leaf = TrainingArguments(
    output_dir="bert_stage2_leaf_base",
    learning_rate=2e-5,
    per_device_train_batch_size=32,   # A100 ok; if slow try 64
    per_device_eval_batch_size=64,
    num_train_epochs=1,
    fp16=True,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    logging_steps=200,
    report_to="none"
)

trainer_leaf_base = WeightedTrainer(
    model=model_leaf_base,
    args=args_leaf,
    train_dataset=ds_leaf_train_base,
    eval_dataset=ds_leaf_test_base,
    tokenizer=tokenizer_leaf
)

trainer_leaf_base.train()
pred_base = trainer_leaf_base.predict(ds_leaf_test_base)
metrics_base = compute_leaf_metrics_from_logits(pred_base.predictions, pred_base.label_ids)
metrics_base


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-multilingual-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
c:\Users\ilker\anaconda3\envs\torchgpu\Lib\site-packages\transformers\training_args.py:1575: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
C:\Users\ilker\AppData\Local\Temp\ipykernel_28268\2151964398.py:21: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `WeightedTrainer.__init__`. Use `processing_class` instead.
  trainer_leaf_base = WeightedTrainer(


Epoch,Training Loss,Validation Loss
1,No log,7.042471


{'acc': 0.0008771929824561404,
 'macro_f1': 1.8409086725207562e-06,
 'weighted_f1': 1.8409086725207562e-06,
 'top3_acc': 0.0035087719298245615,
 'top5_acc': 0.0061403508771929825}

In [21]:
model_leaf_main = AutoModelForSequenceClassification.from_pretrained(
    BASE_MODEL,
    num_labels=len(leaf_label2id),
    id2label=leaf_id2label,
    label2id=leaf_label2id
)

args_leaf_main = TrainingArguments(
    output_dir="bert_stage2_leaf_with_main2",
    learning_rate=2e-5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    num_train_epochs=1,
    fp16=True,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    logging_steps=200,
    report_to="none"
)

trainer_leaf_main = WeightedTrainer(
    model=model_leaf_main,
    args=args_leaf_main,
    train_dataset=ds_leaf_train_main,
    eval_dataset=ds_leaf_test_main,
    tokenizer=tokenizer_leaf
)

trainer_leaf_main.train()
pred_mainfeat = trainer_leaf_main.predict(ds_leaf_test_main)
metrics_mainfeat = compute_leaf_metrics_from_logits(pred_mainfeat.predictions, pred_mainfeat.label_ids)
metrics_mainfeat


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-multilingual-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
c:\Users\ilker\anaconda3\envs\torchgpu\Lib\site-packages\transformers\training_args.py:1575: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
C:\Users\ilker\AppData\Local\Temp\ipykernel_28268\131013167.py:21: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `WeightedTrainer.__init__`. Use `processing_class` instead.
  trainer_leaf_main = WeightedTrainer(


Epoch,Training Loss,Validation Loss
1,No log,7.044122


{'acc': 0.0017543859649122807,
 'macro_f1': 9.472202999499504e-06,
 'weighted_f1': 9.472202999499504e-06,
 'top3_acc': 0.002631578947368421,
 'top5_acc': 0.005263157894736842}

In [ ]:
compare = pd.DataFrame([
    {"model": "Leaf BERT (text only)", **metrics_base},
    {"model": "Leaf BERT (text + predicted MAIN top2)", **metrics_mainfeat},
]).set_index("model")

compare


In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score

LEAF_TRAIN_COL = "category_fixed" if "category_fixed" in train_df.columns else "category"
LEAF_TEST_COL  = "category_fixed" if "category_fixed" in test_df.columns else "category"

# Baseline (SGD) prediction column you already have
SGD_PRED_COL = "pred_leaf_2stage_top2"
assert SGD_PRED_COL in test_df.columns, f"Missing {SGD_PRED_COL} in test_df"

# Try to auto-pick a BERT prediction column from test_df
bert_candidates = [c for c in test_df.columns if ("bert" in c.lower() or "transform" in c.lower()) and ("pred" in c.lower())]
print("BERT pred candidates:", bert_candidates)

# If auto-pick fails, set manually:
BERT_PRED_COL = bert_candidates[0] if len(bert_candidates) == 1 else None
print("Auto-picked BERT_PRED_COL:", BERT_PRED_COL)

# If None, set it explicitly:
# BERT_PRED_COL = "pred_leaf_bert_main2_top1"  # <-- example
assert BERT_PRED_COL in test_df.columns, "Set BERT_PRED_COL to your BERT prediction column name"


In [ ]:
# Train frequency (leaf)
train_freq = train_df[LEAF_TRAIN_COL].astype(str).value_counts()
freq_tbl = train_freq.rename("train_cnt").reset_index().rename(columns={"index": "leaf"})
freq_tbl["cum_cnt"] = freq_tbl["train_cnt"].cumsum()
freq_tbl["cum_share"] = freq_tbl["cum_cnt"] / freq_tbl["train_cnt"].sum()

def slice_name(cum_share):
    if cum_share <= 0.80:
        return "head"
    elif cum_share <= 0.95:
        return "body"
    else:
        return "tail"

freq_tbl["hbt"] = freq_tbl["cum_share"].map(slice_name)

hbt_map = dict(zip(freq_tbl["leaf"], freq_tbl["hbt"]))

freq_tbl.groupby("hbt").agg(
    n_labels=("leaf", "count"),
    train_rows=("train_cnt", "sum")
).assign(train_pct=lambda x: (x["train_rows"] / x["train_rows"].sum() * 100).round(2))


In [ ]:
eval_df = test_df.copy()

eval_df["y_true"] = eval_df[LEAF_TEST_COL].astype(str)
eval_df["y_pred_sgd"] = eval_df[SGD_PRED_COL].astype(str)
eval_df["y_pred_bert"] = eval_df[BERT_PRED_COL].astype(str)

# Map to head/body/tail using train distribution
eval_df["hbt"] = eval_df["y_true"].map(hbt_map).fillna("tail")  # should not happen if no new labels

# Helpful counts
eval_df["hbt"].value_counts(normalize=True).mul(100).round(2)


In [ ]:
def metrics_block(df, y_pred_col):
    y_true = df["y_true"].values
    y_pred = df[y_pred_col].values

    return {
        "n_rows": int(len(df)),
        "n_labels": int(pd.Series(y_true).nunique()),
        "acc": float(accuracy_score(y_true, y_pred)),
        "macro_f1": float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
        "weighted_f1": float(f1_score(y_true, y_pred, average="weighted", zero_division=0)),
    }

rows = []
for part in ["head", "body", "tail", "ALL"]:
    sub = eval_df if part == "ALL" else eval_df[eval_df["hbt"] == part]

    rows.append({"slice": part, "model": "SGD",  **metrics_block(sub, "y_pred_sgd")})
    rows.append({"slice": part, "model": "BERT", **metrics_block(sub, "y_pred_bert")})

res = pd.DataFrame(rows)

# nicer formatting
for c in ["acc", "macro_f1", "weighted_f1"]:
    res[c] = res[c].round(4)

res.sort_values(["slice", "model"])


In [ ]:
pivot = res.pivot(index="slice", columns="model", values=["acc","macro_f1","weighted_f1","n_rows","n_labels"])
display(pivot)

delta = pd.DataFrame({
    "acc_delta": (pivot["acc"]["BERT"] - pivot["acc"]["SGD"]).round(4),
    "macro_f1_delta": (pivot["macro_f1"]["BERT"] - pivot["macro_f1"]["SGD"]).round(4),
    "weighted_f1_delta": (pivot["weighted_f1"]["BERT"] - pivot["weighted_f1"]["SGD"]).round(4),
    "n_rows": pivot["n_rows"]["BERT"].astype(int),
    "n_labels": pivot["n_labels"]["BERT"].astype(int),
})
delta
